### Auto Encoder

In [ ]:
import sys,os
sys.path.append('../../package/')
%run ../../package/init_jupyter.py

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mujoco as mj
import torch as th
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import clear_output, display
from scipy.spatial import distance, procrustes
import json
sys.path.append('../')
sys.path.append('../../asset/')
sys.path.append('../../package/') # for 'preretarget'
sys.path.append('../../package/yoon/')
sys.path.append('../../package/mujoco_helper/')
sys.path.append('../../package/motion_retarget')
sys.path.append('../../package/kinematics_helper/')
sys.path.append('../../package/utility/')
sys.path.append('../../package/smpl/')
sys.path.append('../../package/irvae_tools/')
sys.path.append('../../package/bvh_helper/')
from motion_retarget import *
from geometry import relaxed_distortion_measure
from clustering_robots import *
from _robot_info import *
from mujoco_parser import *
from utils import *
from smpl_model import *
# from smpl_parser import *
from bvh_parser import *

plt.rc('xtick', labelsize=6)
plt.rc('ytick', labelsize=6)
%config InlineBackend.figure_format = 'retina'
%matplotlib inline
print ("PyTorch:[%s]"%(th.__version__))

In [ ]:
def _similarity_transform_umeyama(P, G, with_scale=True):
    """
    P (prediction) and G (GT): shape (J, 3)
    Estimates the similarity transform (s, R, t) aligning P to G with Umeyama (1991),
    Returns the aligned P_hat.
    """
    assert P.shape == G.shape and P.shape[1] == 3
    # transpose to (3, J) to use the standard formula as is
    S1 = P.T   # (3, J)
    S2 = G.T   # (3, J)

    mu1 = S1.mean(axis=1, keepdims=True)
    mu2 = S2.mean(axis=1, keepdims=True)
    X1 = S1 - mu1
    X2 = S2 - mu2

    var1 = np.sum(X1 ** 2)
    K = X1 @ X2.T
    U, s, Vt = np.linalg.svd(K)
    R = Vt.T @ U.T
    # prevent reflection
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    scale = 1.0
    # if with_scale:
    #     scale = np.trace(np.diag(s)) / var1 if var1 > 0 else 1.0
    if with_scale and var1 > 0:
        scale = np.trace(R @ K) / var1     # key fix
    else:
        scale = 1.0

    t = mu2 - scale * R @ mu1
    S1_hat = scale * R @ S1 + t
    P_hat = S1_hat.T  # (J, 3)

    return P_hat, scale, R, t

def _sequence_pa_mpjpe(gt_seq, pred_seq, with_scale=True):
    """
    Align pred -> gt with a similarity transform per frame, compute the mean L2 error over all joints,
    Return the mean over all frames.
    gt_seq, pred_seq: shape (T, J, 3)
    """
    assert gt_seq.shape == pred_seq.shape and gt_seq.shape[2] == 3
    T = gt_seq.shape[0]
    errs = []
    for t in range(T):
        P = pred_seq[t]
        G = gt_seq[t]
        # skip NaN frames
        if not (np.isfinite(P).all() and np.isfinite(G).all()):
            continue
        P_hat, _, _, _ = _similarity_transform_umeyama(P, G, with_scale=with_scale)
        err_t = np.linalg.norm(P_hat - G, axis=1).mean()
        errs.append(err_t)
    return float(np.mean(errs)) if len(errs) > 0 else np.inf

def _sequence_mpjpe(gt_seq, pred_seq, root_strategy='centroid'):
    """
    Similar to Protocol #2: mean L2 error over all joints after removing global translation (default: centering on the joint mean).
    root_strategy: 'centroid' or an integer index (when the joint order is the same).
    """
    assert gt_seq.shape == pred_seq.shape and gt_seq.shape[2] == 3
    T, J, _ = gt_seq.shape
    errs = []
    for t in range(T):
        P = pred_seq[t].copy()
        G = gt_seq[t].copy()
        if not (np.isfinite(P).all() and np.isfinite(G).all()):
            continue
        if root_strategy == 'centroid':
            P -= P.mean(axis=0, keepdims=True)
            G -= G.mean(axis=0, keepdims=True)
        elif isinstance(root_strategy, int) and 0 <= root_strategy < J:
            P -= P[root_strategy:root_strategy+1, :]
            G -= G[root_strategy:root_strategy+1, :]
        else:
            # fallback: centroid
            P -= P.mean(axis=0, keepdims=True)
            G -= G.mean(axis=0, keepdims=True)
        err_t = np.linalg.norm(P - G, axis=1).mean()
        errs.append(err_t)
    return float(np.mean(errs)) if len(errs) > 0 else np.inf

In [ ]:
device = "cuda:0"
device="mps"

# Train Autoencoder

In [ ]:
train=False

In [ ]:
class AutoEncoderClass(nn.Module):
    def __init__(
        self,
        x_dim,
        z_dim=2,
        h_dims=[128, 128],
        loss_type='l2',           # 'l2', 'l1', 'huber'
        actv=nn.ReLU(),           # activation function
        iso = False,
        iso_reg = 1e-7,
    ):
        super(AutoEncoderClass, self).__init__()
        self.name = 'autoencoder'
        self.z_dim = z_dim
        self.h_dims = h_dims
        self.loss_type = loss_type.lower()
        self.actv = actv
        self.iso = iso
        self.iso_reg = iso_reg

        if isinstance(x_dim, int):
            self.input_shape = (x_dim,)
            self.flattened_dim = x_dim
        elif isinstance(x_dim, tuple):
            self.input_shape = x_dim
            self.flattened_dim = int(np.prod(x_dim))
        else:
            raise ValueError("x_dim must be int or tuple")

        enc_layers = []
        in_dim = self.flattened_dim
        for h_dim in self.h_dims:
            enc_layers.append(nn.Linear(in_dim, h_dim))
            enc_layers.append(self.actv)
            in_dim = h_dim
        enc_layers.append(nn.Linear(in_dim, self.z_dim))
        self.encoder = nn.Sequential(*enc_layers)

        dec_layers = []
        in_dim = self.z_dim
        for h_dim in reversed(self.h_dims):
            dec_layers.append(nn.Linear(in_dim, h_dim))
            dec_layers.append(self.actv)
            in_dim = h_dim
        dec_layers.append(nn.Linear(in_dim, self.flattened_dim))
        self.decoder = nn.Sequential(*dec_layers)

    def encode(self, x): return self.encoder(x.view(x.size(0), -1))

    def decode(self, z): return self.decoder(z).view(z.size(0), *self.input_shape)

    def forward(self, x): z = self.encode(x); return self.decode(z), z
    
    def compute_loss(self, x, x_hat):
        if self.loss_type == 'l2': 
            loss = nn.functional.mse_loss(x_hat, x)
        elif self.loss_type == 'l1': 
            loss = nn.functional.l1_loss(x_hat, x)
        elif self.loss_type == 'huber': 
            loss = nn.functional.huber_loss(x_hat, x)
        elif self.loss_type == 'mseloss':
            loss = nn.functional.mse_loss(x_hat,x)
        else: 
            raise ValueError(f"Unknown loss_type: {self.loss_type}")
        if self.iso:
            iso_loss = relaxed_distortion_measure(self.decode, self.encode(x), eta=0.2, metric = 'identity')
            total_loss = loss + self.iso_reg * iso_loss
        else:
            iso_loss = th.tensor(0.0, device=x.device, dtype=loss.dtype)
            total_loss =loss
        return total_loss, loss, iso_loss

# ---------- data loading ----------

def load_data_subset(folder_path):
    arr = np.load(os.path.join(folder_path, 'data.npy'))  # assumes 'data.npy' exists
    print(f"array shape: {arr.shape}")
    if arr.shape[-1]==6:
        data = arr[:, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29], :]
    elif arr.shape[-1]==5:
        data = arr[:, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 21,22,23,24,25,26,27,28,29,30,31], :]
    return data.reshape((30, -1))

def load_all_npy_from_folder(folder_path):
    """
    Load all .npy files in the given folder into a list

    :param folder_path: path to the folder containing .npy files
    :return: list of numpy arrays
    """
    npy_list = []
    for fname in sorted(os.listdir(folder_path)):
        if fname.endswith('.npy'):
            full_path = os.path.join(folder_path, fname)
            arr = np.load(full_path)
            npy_list.append(arr)
    return npy_list

def load_npy_as_dict(folder_path, remove_ext=True):
    """
    Load all .npy files in a folder into a dictionary.

    :param folder_path: Path to folder containing .npy files
    :param remove_ext: Whether to remove '.npy' from the keys
    :return: dict {filename(str): numpy array}
    """
    npy_dict = {}
    for fname in sorted(os.listdir(folder_path)):
        if fname.endswith('.npy'):
            key = fname[:-4] if remove_ext else fname  # remove ".npy"
            full_path = os.path.join(folder_path, fname)
            npy_dict[key] = np.load(full_path)
    return npy_dict

def show_data_matrix(data, cmap='viridis'):
    """
    Show 2D matrix as an image using matshow.
    
    :param data: 2D numpy array (e.g., shape: 30 x 180)
    :param cmap: colormap to use (e.g., 'viridis', 'plasma', 'magma', 'gray')
    """
    plt.figure(figsize=(8, 6))
    plt.matshow(data, cmap=cmap, fignum=1)
    plt.colorbar()
    plt.title("Matrix Visualization (matshow)")
    plt.xlabel("Features")
    plt.ylabel("Samples")
    # plt.tight_layout()
    plt.show()

def save_model(model, path, extra_info=None):
    """
    Save AutoEncoder model with architecture info and optional metadata.

    :param model: AutoEncoderClass instance
    :param path: full path to save file (e.g., 'models/ae_model.pt')
    :param extra_info: optional dict for experiment settings (e.g., loss_type, actv name)
    """
    save_dict = {
        "state_dict": model.state_dict(),
        "x_dim": model.input_shape,
        "z_dim": model.z_dim,
        "h_dims": model.h_dims,
        "loss_type": model.loss_type,
        "actv": model.actv.__class__.__name__,  # store only the name of the activation function
    }

    if extra_info is not None:
        save_dict["extra_info"] = extra_info

    os.makedirs(os.path.dirname(path), exist_ok=True)
    th.save(save_dict, path)
    print(f"Model saved to {path}")

def load_saved_model(model_name, device='cpu'):
    """
    Load an AutoEncoderClass instance from the saved config.json and model.pt.
    
    Args:
        model_name (str): folder name under ../../result/
        device (str): 'cpu', 'cuda:0', etc.
    
    Returns:
        ae (AutoEncoderClass): restored model instance
        config (dict): configuration
    """
    save_dir = os.path.join("../../result", model_name)
    config_path = os.path.join(save_dir, "config.json")
    model_path = os.path.join(save_dir, "model.pt")

    # load config
    with open(config_path, 'r') as f:
        config = json.load(f)

    # initialize the model
    actv_map = {
        'Tanh': th.nn.Tanh(),
        'ReLU': th.nn.ReLU(),
        'SiLU': th.nn.SiLU(),
        'GELU': th.nn.GELU(),
        # add if needed
    }
    actv_fn = actv_map[config['actv']]

    ae = AutoEncoderClass(
        x_dim=config['x_dim'][0],  # should later be set from the data
        z_dim=config['z_dim'],
        h_dims=config['h_dims'],
        loss_type=config['loss_type'],
        actv=actv_fn,
        iso=config['iso'],
        iso_reg=config['iso_reg'],
    ).to(device)

    # load model parameters
    ae.load_state_dict(th.load(model_path, map_location=device))
    ae.eval()

    return ae, config

print ("Ready.")

In [ ]:
# Load data
# repr_kind = 'dh'
repr_kind="screw"
data_path = '../../data/mjcf_test/'+repr_kind + '/'
loaded_data = load_npy_as_dict(data_path)
for k_idx,key in enumerate(loaded_data.keys()):
    print ("[%d/%d] key:[%s] shape:%s"%
           (k_idx,len(loaded_data.keys()),key,loaded_data[key].shape))

In [ ]:
# data = loaded_data['data'][:,[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29],:].reshape((30,-1)) # (30x180)
data = load_data_subset(data_path)
show_data_matrix(data, cmap='viridis')

In [ ]:
def compute_hidden_dims(flattened_dim, z_dim):
    mid1 = int(np.sqrt(flattened_dim * z_dim))
    mid2 = int(np.sqrt(mid1 * z_dim))
    return [mid1, mid2]

# run_experiment lives in package/yoon/ae.py. It never overwrites an existing checkpoint:
# if result/<model_name>/ exists, the new model is saved as <model_name>_<timestamp>.
from ae import run_experiment

# ---- Output location for re-runs ----
# The paper results in ./{easy,middle,hard}_motion_*/ and ./all_*/ are read-only references
# (tutorial 03 reads them). Anything this notebook writes goes to OUT_ROOT instead.
OUT_ROOT = './runs'
def out(rel_path):
    p = os.path.join(OUT_ROOT, rel_path)
    os.makedirs(os.path.dirname(p) or '.', exist_ok=True)
    return p

print ("Ready.")


In [ ]:
if train:
    results, model_name = run_experiment(
        data_np=data,
        h_dims=[64,32],
        z_dim=2,
        num_epochs=1000,
        lr=1e-3,
        actv=nn.Tanh(),
        loss_type='l2',
        device=device,
        batch_size=10,
        iso=True,
        iso_reg=1e-3,
        model_name='dh2'
        )

In [ ]:
robot_chars = RobotCharacteristics()
cur_data_path = os.path.join('../../data/mjcf_test/',repr_kind)
label_data = np.load(os.path.join(cur_data_path,'label.npy'))
njnt_data = np.load(os.path.join(cur_data_path,'njnt.npy'))
indices_data = np.load(os.path.join(cur_data_path,'indices.npy'))
robot_chars.max_joints_per_group = njnt_data

SMPL

In [ ]:
# pkl_path = '../../data/smpl_model/model.pkl'
# file_path = '../../data/mdm_motion/class3/samples_humanml_enc_512_50steps_000750000_seed10_Dance_the_Samba_with_quick_foot_steps_and_rhythmic_arm_swings/sample00_rep01_smpl_params.npy'
# motion_type = 'smpl'

CMU

In [ ]:
pkl_path = None,
# file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/141/141_16.bvh' # easy
file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/018/18_15.bvh' # middle
# file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/079/79_02.bvh' # hard
motion_type = 'cmu'

In [ ]:
def sum_without_negatives(disparity):
    # sum only the elements other than -999
    total_sum = sum(value for value in disparity if value != -999)

    return total_sum

def get_smpl_pos(chains,joi_body_name,base_between_pelvis=True):
    arrays=[]
    joi_body_name_key = list(joi_body_name.keys())
    joi_body_name_key = [joi.replace('spine','spine1')for joi in joi_body_name_key]
    for i in range(len(chains)):
        smallest_arrays=[]
        for j in range(len(joi_body_name_key)):
            if joi_body_name_key[j] in chains[i].joint_names:
                if base_between_pelvis and joi_body_name_key[j]=='base':
                    p_rp,_ = chains[i].get_joint_pR(joint_name='rp')
                    p_lp,_ = chains[i].get_joint_pR(joint_name='lp')
                    p = 0.5*(p_rp+p_lp)
                else:
                    p,_ = chains[i].get_joint_pR(joint_name=joi_body_name_key[j])
                smallest_arrays.append(p)
        small_arrays=np.array(smallest_arrays)
        arrays.append(small_arrays[np.newaxis,:,:])
    result=np.vstack(arrays)
    return result

def get_cmu_pos(chains,joi_body_name,base_between_pelvis=True, fix_root = False,):
    pairs = {
    'base' : 'Hips',
    'spine' : 'Spine',
    'rs' : 'RightArm',
    're' : 'RightForeArm',
    'rw' : 'RightHand',
    'rh' : 'RightHand',
    'ls' : 'LeftArm',
    'le' : 'LeftForeArm',
    'lw' : 'LeftHand',
    'lh' : 'LeftHand',
    'neck' : 'Neck',
    'head' : 'Head',
    'rp' : 'RightUpLeg',
    'rk' : 'RightLeg',
    'ra' : 'RightFoot',
    'rf' : 'RightFoot',
    'lp' : 'LeftUpLeg',
    'lk' : 'LeftLeg',
    'la' : 'LeftFoot',
    'lf' : 'LeftFoot',
    'rtoe' : 'RightToeBase',
    'ltoe' : 'LeftToeBase',
             }
    arrays=[]
    
    joi_body_name_key = list(joi_body_name.keys())
    for i in range(len(chains)):
        smallest_arrays=[]
        chain = chains[i]
        if fix_root:
            p_root,R_root = chain.get_root_joint_pR()
            chain_origin = deepcopy(chain)
            chain_origin.set_root_joint_pR(p=np.array([0,0,0],R=R_root))
            chain_origin.forward_kinematics()
    
        for j in range(len(joi_body_name_key)):
            
            if joi_body_name_key[j]=='root':
                p_rp,_ = chains[i].get_joint_pR(joint_name=pairs['rp'])
                p_lp,_ = chains[i].get_joint_pR(joint_name=pairs['lp'])
                p = 0.5*(p_rp+p_lp)
                continue
            elif pairs[joi_body_name_key[j]] in chains[i].joint_names:
                if base_between_pelvis and joi_body_name_key[j]=='base':
                    p_rp,_ = chains[i].get_joint_pR(joint_name=pairs['rp'])
                    p_lp,_ = chains[i].get_joint_pR(joint_name=pairs['lp'])
                    p = 0.5*(p_rp+p_lp)
                elif joi_body_name_key[j]=='neck':
                    p_neck,_ = chains[i].get_joint_pR(joint_name=pairs[joi_body_name_key[j]])
                    p_rs,_ = chains[i].get_joint_pR(joint_name=pairs['rs'])
                    p_ls,_ = chains[i].get_joint_pR(joint_name=pairs['ls'])
                    p_neck_z = 0.5*(p_rs+p_ls) 
                    p = np.array([p_neck[0],p_neck[1],p_neck_z[2]])
                else:
                    p,_ = chains[i].get_joint_pR(joint_name=pairs[joi_body_name_key[j]])
            if fix_root:
                p_base,_=chains[i].get_joint_pR(joint_name='Hips')
                p-=p_base
                smallest_arrays.append(p)
                
            else:
                smallest_arrays.append(p)
            # smallest_arrays.append(p)        
        small_arrays=np.array(smallest_arrays)
        # print(f"small arrays: {small_arrays.shape}")
        arrays.append(small_arrays[np.newaxis,:,:])
    result=np.vstack(arrays)
    return result

def r_sq(x1,x2,x_range=1.0,invlen=5.0):
    """
    Scaled pairwise dists 
    """
    x1_scaled,x2_scaled = invlen*x1/x_range,invlen*x2/x_range
    D_sq = distance.cdist(x1_scaled,x2_scaled,'sqeuclidean') 
    return D_sq

# def retarget_and_procrustes(
#         env,
#         pkl_path,
#         file_path,
#         motion_type = 'smpl', # 'cmu', 'smpl'
#         ik_p_spine=False,
#         ik_p_neck=True,
#         leg_fixed=False,
#         ik_R_hand=False,

#         animate_chains = False,
#         animate_ik=False,
#         verbose=False,
#         override = None
#                   ):
#         # Load SMPL model

#     joi_body_name=get_joi_body_name_of_irae(override=override)
#     joi_smpl_name = list(joi_body_name.keys()) #smpl joi name
#     joi_robot_name = list(joi_body_name.values()) # robot body name of joi
#     if motion_type=='smpl':
#         smpl = SMPLModelClass(model_path=pkl_path)
#         # Given file path
    
#         # Extracting the base name of the file
#         file_name = os.path.basename(file_path)

#         # Extract the motion_name
#         # Since file_name does not include '/', directly use split on file_path before extracting the file name
#         motion_name = file_path.split('seed10_')[1].split('/')[0] if 'seed10_' in file_path else None

#         # Extract the sample
#         sample = file_name.split('sample00_')[1].split('_smpl')[0] if 'sample00_' in file_name else None

#         # Extract the class_name from file_path
#         class_name = file_path.split('mdm_motion/')[1].split('/samples_')[0] if 'mdm_motion/' in file_path and '/samples_' in file_path else None
#         if verbose:
#             print(f"motion_name: {motion_name}")
#             print(f"sample: {sample}")
#             print(f"class_name: {class_name}")
        
#         secs, chains = get_chains_from_smpl(
#             file_path        = file_path,
#             smpl             = smpl,
#             intp_traj        = False,
#             plot_chain_graph = False)
#         # print(f"motion_type: {motion_type}\n chains: {chains}")
#         motion_pos=get_smpl_pos(chains,joi_body_name)
#     elif motion_type=='cmu':
#         secs, chains = get_chains_from_bvh_cmu_mocap(
#             bvh_path = file_path,
#             p_rate = 0.056444*1.0,
#             zup=True,
#             plot_chain_graph = False,
#             verbose = False,
#         )
#         # print(f"motion_type: {motion_type}\n chains: {chains}")
#         motion_pos=get_cmu_pos(chains,joi_body_name)
    
#     L = len(chains); dt = secs[1]-secs[0]; HZ = int(1/dt)
    

#     if animate_chains:
#         animate_chains_slider(env,secs,chains)
    
#     if verbose:
#         print ("[%s] loaded. L:[%d] dt:[%.4f]sec HZ:[%d]"%(file_path,L,dt,HZ))

#     # preretarget
#     qpos_dir_list = directional_preretarget_postrigging(
#     env,
#     secs,
#     chains,
#     ik_p_spine=ik_p_spine,
#     ik_p_neck=ik_p_neck,
#     leg_fixed=leg_fixed,
#     src_name           = motion_type,
#     trgt_name          = 'irae',
#     joint_names_for_ik = None,
#     max_ik_tick        = 100, # maximum IK iteration
#     ik_stepsize        = 1, # IK stepsize
#     ik_err_th          = 1e-3, # IK error threshold
#     print_ik_failure   = False,
#     ik_R_hand          = ik_R_hand,
#     ik_R_foot          = not leg_fixed,
#     reset_env          = True,
#     animate            = animate_ik, # animate pre-retarget process
#     plot_joi_src       = False,
#     plot_joi_trgt      = False,
#     render_every       = 1,
#     verbose            = verbose,
#     print_every        = 30,
#     pause_each_tick    = False,
#     post_rigging       = True,
#     override           = override,
#     )
#     # Initialize MuJoCo viewer
#     env.reset(step=True)
#     if animate_ik:
#         env.init_viewer(
#             title='MDM to auto_gen Pre-Retarget',
#             transparent=True)

#     motion_length = L
#     num_bodies = env.n_body

#     robot_joi_pos = np.zeros((motion_length,len(joi_robot_name),3))
#     for tick in range(motion_length):
#         env.forward(q=qpos_dir_list[tick]) # FK
#         for body_idx in range(num_bodies):
#             if body_idx == 0:
#                 continue
#             body_name = env.body_names[body_idx]
            
#             if body_name in joi_robot_name:
#                 # Get the current position of the body
#                 joi_smpl_idx = [i for i, value in enumerate(joi_robot_name) if value == body_name]
#                 for i in range(len(joi_smpl_idx)):
#                     body_pos = env.data.body(body_name).xpos
#                     robot_joi_pos[tick,joi_smpl_idx[i],:]=body_pos

#     """ SMPL JOIs have no root! + Procrustes analysis must be run per body
#         evaluate smpl_pos(motion_length,11,3) against robot_joi_pos(motion_length,12,3) per JOI
#     """

#     # drop the root (:,0,:) of robot_joi_pos to match smpl_pos, giving shape (120,11,3)
#     robot_joi_pos_wo_root = np.delete(robot_joi_pos, 0, axis=1) # drop element 0 along axis 1
#     _ = joi_smpl_name.pop(0)
#     procrustes_results = []

#     # # run Procrustes analysis

#     """Assuming this is done per joint"""
#     for i in range(motion_pos.shape[1]):
#         matrix1 = motion_pos[:, i, :]
#         matrix2 = robot_joi_pos_wo_root[:, i, :]
        
#         try:
#             # run Procrustes analysis for each i
#             mtx1, mtx2, disparity = procrustes(matrix1, matrix2)
#             procrustes_results.append((mtx1, mtx2, disparity))
#         except:
#             procrustes_results.append((matrix1, matrix2, -999))
        
#     result = []
#     # print example results
#     for i, (mtx1, mtx2, disparity) in enumerate(procrustes_results):
#         result.append(disparity)
#         if verbose:
#             print(f"Procrustes analysis for column {i} - joi name: {joi_smpl_name[i]} - Disparity: {disparity:.3f}")
#             print("disparity: ", result)

#             print("Done.")
#     if animate_ik:
#         env.viewer.close()
#     return sum_without_negatives(result)

def retarget_and_procrustes(
        env,
        pkl_path,
        file_path,
        motion_type = 'smpl', # 'cmu', 'smpl'
        ik_p_spine=False,
        ik_p_neck=True,
        leg_fixed=False,
        ik_R_hand=False,

        animate_chains = False,
        animate_ik=False,
        verbose=False,
        override = None,
        eval="procrustes",          # <-- added: "procrustes" | "PA-MPJPE"
        jnt_penalty=True,
        jnt_coeff=1,
        unit_scale=1000,             # <-- added: coordinate unit scale (e.g. 1000 for m -> mm)
        p2_root_strategy='centroid' # <-- added: MPJPE (root removed) mode
    ):
    """
    Returns:
      - eval="procrustes": sum of per-joint time-series Procrustes disparities (same as before)
      - eval="PA-MPJPE" : sequence-mean PA-MPJPE (unit_scale applied)
    """

    # Load SMPL/CMU motion & joint mapping
    joi_body_name = get_joi_body_name_of_irae(override=override)
    joi_smpl_name = list(joi_body_name.keys())   # smpl joi name
    joi_robot_name = list(joi_body_name.values())# robot body name of joi

    if motion_type=='smpl':
        smpl = SMPLModelClass(model_path=pkl_path)

        file_name  = os.path.basename(file_path)
        motion_name = file_path.split('seed10_')[1].split('/')[0] if 'seed10_' in file_path else None
        sample      = file_name.split('sample00_')[1].split('_smpl')[0] if 'sample00_' in file_name else None
        class_name  = file_path.split('mdm_motion/')[1].split('/samples_')[0] if 'mdm_motion/' in file_path and '/samples_' in file_path else None
        if verbose:
            print(f"motion_type: {motion_type}, motion_name: {motion_name}, sample: {sample}, class_name: {class_name}")

        secs, chains = get_chains_from_smpl(
            file_path        = file_path,
            smpl             = smpl,
            intp_traj        = False,
            plot_chain_graph = False)
        motion_pos = get_smpl_pos(chains, joi_body_name)   # (T, J_smpl, 3)

    elif motion_type=='cmu':
        secs, chains = get_chains_from_bvh_cmu_mocap(
            bvh_path = file_path,
            p_rate   = 0.056444*1.0,
            zup      = True,
            plot_chain_graph = False,
            verbose  = False,
        )
        motion_pos = get_cmu_pos(chains, joi_body_name)    # (T, J_smpl, 3)

    L = len(chains); dt = secs[1]-secs[0]; HZ = int(1/dt)

    if animate_chains:
        animate_chains_slider(env, secs, chains)

    if verbose:
        print(f"[{file_path}] loaded. L:{L}  dt:{dt:.4f}s  HZ:{HZ}")

    # Pre-retarget (original logic kept)
    qpos_dir_list = directional_preretarget_postrigging(
        env,
        secs,
        chains,
        ik_p_spine=ik_p_spine,
        ik_p_neck=ik_p_neck,
        leg_fixed=leg_fixed,
        src_name           = motion_type,
        trgt_name          = 'irae',
        joint_names_for_ik = None,
        max_ik_tick        = 100,
        ik_stepsize        = 1,
        ik_err_th          = 1e-3,
        print_ik_failure   = False,
        ik_R_hand          = ik_R_hand,
        ik_R_foot          = not leg_fixed,
        reset_env          = True,
        animate            = animate_ik,
        plot_joi_src       = False,
        plot_joi_trgt      = False,
        render_every       = 1,
        verbose            = verbose,
        print_every        = 30,
        pause_each_tick    = False,
        post_rigging       = True,
        override           = override,
    )

    # collect robot joint positions from the simulator
    env.reset(step=True)
    if animate_ik:
        env.init_viewer(title='MDM to auto_gen Pre-Retarget', transparent=True)

    motion_length = L
    num_bodies = env.n_body
    robot_joi_pos = np.zeros((motion_length, len(joi_robot_name), 3), dtype=np.float32)

    for tick in range(motion_length):
        env.forward(q=qpos_dir_list[tick])  # FK
        for body_idx in range(1, num_bodies):  # body 0 is world/root, usually skipped
            body_name = env.body_names[body_idx]
            if body_name in joi_robot_name:
                jj = [i for i, v in enumerate(joi_robot_name) if v == body_name]
                for j in jj:
                    robot_joi_pos[tick, j, :] = env.data.body(body_name).xpos

    # === prepare evaluation: SMPL has no root, so drop the robot root (index 0) to match dimensions ===
    robot_joi_pos_wo_root = np.delete(robot_joi_pos, 0, axis=1)     # (T, J, 3)
    joi_smpl_name_wo_root = joi_smpl_name[1:]                        # (J,)  <-- slicing instead of pop() (keeps the original)
    # test item: base is 0 anyway (upper body, so the base does not move), so drop the base too
    # robot_joi_pos_wo_root = np.delete(robot_joi_pos, (0,1), axis=1)     # (T, J, 3)
    # joi_smpl_name_wo_root = joi_smpl_name[2:]                        # (J,)  <-- slicing instead of pop() (keeps the original)
    # motion_pos = motion_pos[:,1:,:]
    # check that shapes match
    assert motion_pos.shape[:2] == robot_joi_pos_wo_root.shape[:2], \
        f"Shape mismatch: GT {motion_pos.shape} vs Pred {robot_joi_pos_wo_root.shape}"

    # =========================
    # evaluation branch
    # =========================
    eval_key = str(eval).lower()
    if eval_key == "procrustes":
        # original method: align each joint's time series (T x 3) with Procrustes and compute the disparity per joint
        procrustes_results = []
        J = motion_pos.shape[1]
        for j in range(J):
            matrix1 = motion_pos[:, j, :]              # (T, 3)
            matrix2 = robot_joi_pos_wo_root[:, j, :]   # (T, 3)
            try:
                _, _, disparity = procrustes(matrix1, matrix2)
                procrustes_results.append(disparity)
            except Exception as e:
                # if verbose:
                print(f"[Procrustes] joint {j} ({joi_smpl_name_wo_root[j]}) failed: {e}")
                procrustes_results.append(-999.0)

        if verbose:
            for j, d in enumerate(procrustes_results):
                print(f"Procrustes joint {j} ({joi_smpl_name_wo_root[j]}): disparity={d:.6f}")

        if animate_ik:
            env.viewer.close()

        # keep the original return rule: sum excluding -999
        result = float(sum(d for d in procrustes_results if d != -999.0))

    elif eval_key in ("pa-mpjpe", "p-mpjpe", "p1"):
        # Human3.6M Protocol #1 (PA-MPJPE): per-frame similarity alignment over all joints, then mean L2
        pa_mpjpe = _sequence_pa_mpjpe(
            gt_seq   = motion_pos,
            pred_seq = robot_joi_pos_wo_root,
            with_scale=True
        ) * unit_scale

        # for reference, also compute Protocol #2 (similar to root-removed MPJPE) and log it
        p2_mpjpe = _sequence_mpjpe(
            gt_seq   = motion_pos,
            pred_seq = robot_joi_pos_wo_root,
            root_strategy=p2_root_strategy
        ) * unit_scale

        if verbose:
            unit_str = " (scaled)" if unit_scale != 1.0 else ""
            print(f"PA-MPJPE (Protocol #1){unit_str}: {pa_mpjpe:.3f}")
            print(f"MPJPE   (root={p2_root_strategy}){unit_str}: {p2_mpjpe:.3f}")

        if animate_ik:
            env.viewer.close()

        # the default return is the PA-MPJPE value (a scalar, for comparison)
        result =  float(pa_mpjpe)

    else:
        if animate_ik:
            env.viewer.close()
        raise ValueError("`eval` must be one of {'procrustes', 'PA-MPJPE'}")
    
    if jnt_penalty:
        n_joint = env.n_joint
        # print(f"n_joint: {n_joint}")
        return result + jnt_coeff * n_joint

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial import distance

np.random.seed(seed=1)
def override_joi(env,repr_kind):
    body_names = env.body_names
    
    # keep the names to match in each group
    group_map = {
        'base_link': [],
        'torso': [],
        'neck': [],
        'right_shoulder_girdle': [],
        'right_shoulder': [],
        'right_upperarm': [],
        'right_elbow': [],
        'right_forearm': [],
        'right_wrist': [],
        'left_shoulder_girdle': [],
        'left_shoulder': [],
        'left_upperarm': [],
        'left_elbow': [],
        'left_forearm': [],
        'left_wrist': [],
    }

    # classify names
    for name in body_names:
        for group in group_map.keys():
            if group in name:
                group_map[group].append(name)

    # set the fallback path
    fallback_order_screw = {
        'right_elbow': ['right_upperarm', 'right_shoulder', 'right_shoulder_girdle', 'torso', 'base_link'],
        'right_wrist': ['right_forearm', 'right_elbow', 'right_upperarm', 'right_shoulder', 'right_shoulder_girdle', 'torso', 'base_link'],
        'left_elbow': ['left_upperarm', 'left_shoulder', 'left_shoulder_girdle', 'torso', 'base_link'],
        'left_wrist': ['left_forearm', 'left_elbow', 'left_upperarm', 'left_shoulder', 'left_shoulder_girdle', 'torso', 'base_link'],
        'torso': ['base_link'],
        'neck': ['torso', 'base_link'],
        'right_shoulder': ['right_shoulder_girdle', 'torso', 'base_link'],
        'left_shoulder': ['left_shoulder_girdle', 'torso', 'base_link'],
    }
    fallback_order_dh = {
        'right_elbow': ['right_upperarm', 'right_shoulder', 'right_shoulder_girdle',  'torso', 'base_link'],
        'right_tcp': ['right_wrist','right_forearm', 'right_elbow', 'right_upperarm', 'right_shoulder', 'right_shoulder_girdle',  'torso', 'base_link'],
        'left_elbow': ['left_upperarm', 'left_shoulder', 'left_shoulder_girdle',  'torso', 'base_link'],
        'left_tcp': ['left_wrist','left_forearm', 'left_elbow', 'left_upperarm', 'left_shoulder', 'left_shoulder_girdle',  'torso', 'base_link'],
        'torso': ['base_link'],
        'neck': ['torso', 'base_link'],
        'right_shoulder': ['right_shoulder_girdle',  'torso', 'base_link'],
        'left_shoulder': ['left_shoulder_girdle',  'torso', 'base_link'],
    }
    if repr_kind=="screw":
        fallback_order=fallback_order_screw
    elif repr_kind=="dh":
        fallback_order=fallback_order_dh
    targets = ['torso', 'neck', 'right_shoulder', 'right_elbow', 'right_wrist',
               'left_shoulder', 'left_elbow', 'left_wrist']
    result_name    = ['spine', 'neck', 'rs', 're', 'rw', 'ls', 'le', 'lw']

    results = {}

    for idx, group in enumerate(targets):
        # find directly
        matches = group_map.get(group, [])
        joi_type = result_name[idx]
        if matches:
            # selected = matches[int(len(matches) / 2)]
            # selected = matches[0]
            selected = matches[-1]
            results[joi_type] = selected
            if joi_type =='rw':
                results['rh'] = selected
            if joi_type =='lw':
                results['lh'] = selected
            continue

        # take the fallback
        fallback_list = fallback_order.get(group, [])
        found = False
        for fallback in fallback_list:
            candidates = group_map.get(fallback, [])
            if candidates:
                selected = candidates[-1]  # last one
                results[joi_type] = selected
                found = True
                if joi_type =='rw':
                    results['rh'] = selected
                if joi_type =='lw':
                    results['lh'] = selected
                break
        if not found:
            continue
    return results

def evaluate(x,
             model,
             robot_chars, 
             motion_type,
             pkl_path,
             file_path,
             animate=False,
             repr_kind="screw",
             eval="procrustes",
             jnt_penalty=True,
             jnt_coeff=1,
             verbose=False):
    results= np.zeros((x.shape[0],1))
    decode = model.decode(th.tensor(x,device=next(model.parameters()).device,dtype=th.float32)).detach().numpy()
    full_decode=robot_chars.build_full_interp_from_symmetric(interp_data_sym=decode,threshold=0.5,repr_kind=repr_kind)
    data_to_visualize = robot_chars.convert_flat_interp_to_dict(flat_interp_data = full_decode,repr_kind=repr_kind)
    # print(f"data_to_vis: {data_to_visualize}")
    robot_chars.visualize_data_to_classify(data_to_visualize, save_video=False,vis=False, xml=True, repr_kind = repr_kind)
    i=0
    for name, xml in robot_chars.xml.items():
        env = MuJoCoParserClass(name=name, xml_string=xml, verbose=False)
        if len(env.get_idxs_jac(joint_names=env.rev_joint_names))<=5:
            if eval in ("pa-mpjpe", "p-mpjpe", "p1"):
                results[i]=300.0
            elif eval=="procrustes":
                results[i]=10.0
            i+=1
            continue
        things_to_override = override_joi(env,repr_kind)
        # print(env.body_names)
        result = retarget_and_procrustes(
                    env,
                    pkl_path=pkl_path,
                    file_path=file_path,
                    motion_type = motion_type,
                    ik_p_spine=True,
                    ik_p_neck=False,
                    leg_fixed=True,
                    ik_R_hand=False,
                    animate_chains = False,
                    animate_ik=animate,
                    verbose=False,
                    override= things_to_override,
                    eval=eval,
                    jnt_penalty=jnt_penalty,
                    jnt_coeff=jnt_coeff,
                    )
        # if i%500==499: print(f"i: {i} result: {result}")
        if verbose: print(f"{x[i]} result: {result}")
        results[i]=result
        i+=1
    return results

def r_sq(x1, x2, x_range=1.0, invlen=5.0):
    x1_scaled, x2_scaled = invlen * x1 / x_range, invlen * x2 / x_range
    return distance.cdist(x1_scaled, x2_scaled, 'sqeuclidean')

In [ ]:
p_global = 0.55                 # previously 0.5 -> a bit more global exploration
length_scale = 3.5              # rugged landscape: short length scale
invlen = 1.0 / (length_scale**2)  # = 0.0816
n_loop_switch = 20              # previously 10 -> probe more global candidates
max_loop = int(5e2)             # previously 1e3 -> 500 (less wasted time)
sigma_coef = 0.6                # local proposal width coefficient
eval = "PA-MPJPE"
jnt_penalty=True
jnt_coeff = 3.5
animate=False
master_seed = 123456789  # any number works. Just record it in the experiment log!
n_runs = 10
ss = np.random.SeedSequence(master_seed)
child_ss = ss.spawn(n_runs)                     # 10 independent seed branches
rngs = [np.random.default_rng(cs) for cs in child_ss]  # one RNG per run

In [ ]:
model_names={
'2dim_isoTrue':'h64_32_z2_isoTrue_actvTanh_isoreg1e-07_nepoch1000_bs10',
'2dim_isoFalse':'h64_32_z2_isoFalse_actvTanh_isoreg1e-07_nepoch1000_bs10',
'3dim_isoTrue':'h64_32_z3_isoTrue_actvTanh_isoreg1e-07_nepoch1000_bs10'}
# results={}
# for name, model_name in enumerate(model_names):
#     ae, config = load_saved_model(model_name=model_name, device="cpu")
#     # Range
#     x_dim = ae.z_dim
#     x_minval, x_maxval = -10,10

#     # collect initial data
#     x_voo = x_minval + (x_maxval - x_minval) * np.random.rand(16, x_dim)
#     # print(f"x_voo: {x_voo}")
#     y_voo = evaluate(x_voo,ae, robot_chars, motion_type,
#                     pkl_path,file_path,animate=animate,repr_kind=repr_kind,
#                     eval=eval,jnt_penalty=jnt_penalty,jnt_coeff=jnt_coeff,verbose=True)
#     # print(f"y_voo: {y_voo}")

#     max_iter = 10
#     for it in range(max_iter):

#         # (1) only replace the global/local branch probability
#         if np.random.rand() < p_global:   # previously: < 0.5
#             x_sel = x_minval + (x_maxval - x_minval) * np.random.rand(1, x_dim)
#         else:
#             idx_min = np.argmin(y_voo)
#             x_min_pt = x_voo[idx_min, :].reshape((1, -1))

#             # (2) only replace invlen
#             n = x_voo.shape[0] - 1
#             dists = r_sq(x_min_pt, np.delete(x_voo, [idx_min], axis=0), invlen=invlen)  # previously: invlen=1.0
#             dist_min = np.min(dists)

#             n_loop = 0
#             while True:
#                 n_loop += 1
#                 if n_loop < n_loop_switch:   # previously: < 10
#                     x_sel = x_minval + (x_maxval - x_minval) * np.random.rand(1, x_dim)
#                 else:
#                     # (3) scale only the local proposal width + clip to bounds
#                     sigma = sigma_coef * np.sqrt(max(dist_min, 1e-12) / float(x_dim))
#                     x_sel = x_min_pt + sigma * np.random.randn(1, x_dim)
#                     x_sel = np.clip(x_sel, x_minval, x_maxval)
#                 dist_sel = r_sq(x_min_pt, x_sel, invlen=invlen)
#                 if dist_sel < dist_min:
#                     break
#                 if n_loop >= max_loop:       # previously: 1e3
#                     # on failure, take one global sample and exit
#                     x_sel = x_minval + (x_maxval - x_minval) * np.random.rand(1, x_dim)
#                     break

#         # evaluation/update part unchanged
#         y_sel = evaluate(x_sel, model=ae, robot_chars=robot_chars, motion_type=motion_type,
#                         pkl_path=pkl_path, file_path=file_path, animate=animate,repr_kind=repr_kind,
#                         eval=eval,jnt_penalty=jnt_penalty,jnt_coeff=jnt_coeff,verbose=True)
#         # print(f"y_sel: {y_sel}")
#         x_voo = np.concatenate((x_voo, x_sel), axis=0)
#         y_voo = np.concatenate((y_voo, y_sel), axis=0)
#         print(f"Optimal idx: {np.argmin(y_voo)}")
#     # store only y, in order
#     results[name] = np.asarray(y_voo, dtype=float).ravel()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

# ── 1) run once and return only the y sequence ─────────────────────────────────────────
def run_once(model_spec, rng):
    # fix all possible random sources (the evaluation function may use np/random/torch internally)

    ae, config = load_saved_model(model_name=model_spec, device="cpu")
    x_dim = ae.z_dim
    x_minval, x_maxval = -15, 15

    # use rng instead of np.random
    rand = rng.random
    randn = rng.standard_normal

    # initial samples / evaluation
    x_voo = x_minval + (x_maxval - x_minval) * rand((16, x_dim))
    y_voo = evaluate(x_voo, ae, robot_chars, motion_type,
                     pkl_path, file_path, animate=animate, repr_kind=repr_kind,
                     eval=eval, jnt_penalty=jnt_penalty, jnt_coeff=jnt_coeff, verbose=False)

    max_iter = 100
    for it in range(max_iter):
        if rand(1)[0] < p_global:
            x_sel = x_minval + (x_maxval - x_minval) * rand((1, x_dim))
        else:
            idx_min = np.argmin(y_voo)
            x_min_pt = x_voo[idx_min, :].reshape((1, -1))
            n = x_voo.shape[0] - 1
            dists = r_sq(x_min_pt, np.delete(x_voo, [idx_min], axis=0), invlen=invlen)
            dist_min = np.min(dists)

            n_loop = 0
            while True:
                n_loop += 1
                if n_loop < n_loop_switch:
                    x_sel = x_minval + (x_maxval - x_minval) * rand((1, x_dim))
                else:
                    sigma = sigma_coef * np.sqrt(max(dist_min, 1e-12) / float(x_dim))
                    x_sel = x_min_pt + sigma * randn((1, x_dim))
                    x_sel = np.clip(x_sel, x_minval, x_maxval)
                dist_sel = r_sq(x_min_pt, x_sel, invlen=invlen)
                if dist_sel < dist_min or n_loop >= max_loop:
                    if n_loop >= max_loop:  # on failure, one global sample
                        x_sel = x_minval + (x_maxval - x_minval) * rand((1, x_dim))
                    break

        y_sel = evaluate(x_sel, model=ae, robot_chars=robot_chars, motion_type=motion_type,
                         pkl_path=pkl_path, file_path=file_path, animate=animate, repr_kind=repr_kind,
                         eval=eval, jnt_penalty=jnt_penalty, jnt_coeff=jnt_coeff, verbose=False)
        x_voo = np.concatenate((x_voo, x_sel), axis=0)
        y_voo = np.concatenate((y_voo, y_sel), axis=0)

    return np.asarray(x_voo,dtype=float).ravel(),np.asarray(y_voo, dtype=float).ravel()

# ── 2) model/seed setup ───────────────────────────────────────────────────────
# model_names = {
#     '2dim_isoTrue' : 'h64_32_z2_isoTrue_actvTanh_isoreg1e-07_nepoch1000_bs10',
#     '2dim_isoFalse': 'h64_32_z2_isoFalse_actvTanh_isoreg1e-07_nepoch1000_bs10',
#     '3dim_isoTrue' : 'h64_32_z3_isoTrue_actvTanh_isoreg1e-07_nepoch1000_bs10',
# }
# -- abstract) for generating optimization data per latent dimension (use the one above for the original experiments) --
model_names = {
    f"{z}dim_isoTrue": f"h64_32_z{z}_isoTrue_actvTanh_isoreg1e-07_nepoch1000_bs10" for z in range(5, 121)
}

# (optional) to use a completely different seed set per model, uncomment below and ignore the seeds above:
# base_seed_per_model = {'2dim_isoTrue':100000, '2dim_isoFalse':200000, '3dim_isoTrue':300000}

# ── 3) run ────────────────────────────────────────────────────────────────
runs_by_model = {k: [] for k in model_names.keys()}
for tag, spec in model_names.items():
    for i in range(n_runs):
        # use shared seeds:
        rng=rngs[i]
        # (optional) to use independent seeds per model:
        # seed = base_seed_per_model[tag] + i

        x_seq,y_seq = run_once(spec, rng)
        runs_by_model[tag].append(y_seq)
        
    # save each one individually (added for the abstract)
    np.save(out(tag+'VOO.npy'),runs_by_model[tag])
    runs_by_model = {k: [] for k in model_names.keys()} # save and reset per model to save memory
    print(f"Completed runs for model: {tag}")

for key, value in enumerate(runs_by_model):
    print(f"keys: {key}, values: {value}")
    np.save(out(value+'VOO.npy'),runs_by_model[value])


In [ ]:
def evaluate_raw(x,
             robot_chars, 
             motion_type,
             pkl_path,
             file_path,
             animate=False,
             repr_kind="screw",
             eval="procrustes",
             jnt_penalty=True,
             jnt_coeff=1,
             verbose=False):
    results= np.zeros((x.shape[0],1))
    # decode = model.decode(th.tensor(x,device=next(model.parameters()).device,dtype=th.float32)).detach().numpy()
    full_decode=robot_chars.build_full_interp_from_symmetric(interp_data_sym=x,threshold=0.5,repr_kind=repr_kind)
    data_to_visualize = robot_chars.convert_flat_interp_to_dict(flat_interp_data = full_decode,repr_kind=repr_kind)
    # print(f"data_to_vis: {data_to_visualize}")
    robot_chars.visualize_data_to_classify(data_to_visualize, save_video=False,vis=False, xml=True, repr_kind = repr_kind)
    i=0
    for name, xml in robot_chars.xml.items():
        env = MuJoCoParserClass(name=name, xml_string=xml, verbose=False)
        if len(env.body_names)<=2:
            results[i]=10.0
            i+=1
            continue
        things_to_override = override_joi(env,repr_kind)
        # print(env.body_names)
        result = retarget_and_procrustes(
                    env,
                    pkl_path=pkl_path,
                    file_path=file_path,
                    motion_type = motion_type,
                    ik_p_spine=True,
                    ik_p_neck=False,
                    leg_fixed=True,
                    ik_R_hand=False,
                    animate_chains = False,
                    animate_ik=animate,
                    verbose=False,
                    override= things_to_override,
                    eval=eval,
                    jnt_penalty=jnt_penalty,
                    jnt_coeff=jnt_coeff,
                    )
        # if i%500==499: print(f"i: {i} result: {result}")
        if verbose: print(f"{x[i]} result: {result}")
        results[i]=result
        i+=1
    return results
def run_raw_space_once(rng):
    # fix all possible random sources (the evaluation function may use np/random/torch internally)

    # ae, config = load_saved_model(model_name=model_spec, device="cpu")
    x_dim = 120
    x_minval, x_maxval = -15, 15

    # use rng instead of np.random
    rand = rng.random
    randn = rng.standard_normal

    # initial samples / evaluation
    x_voo = x_minval + (x_maxval - x_minval) * rand((16, x_dim))
    y_voo = evaluate_raw(x_voo, robot_chars, motion_type,
                     pkl_path, file_path, animate=animate, repr_kind=repr_kind,
                     eval=eval, jnt_penalty=jnt_penalty, jnt_coeff=jnt_coeff, verbose=False)

    max_iter = 30
    for it in range(max_iter):
        if rand(1)[0] < p_global:
            x_sel = x_minval + (x_maxval - x_minval) * rand((1, x_dim))
        else:
            idx_min = np.argmin(y_voo)
            x_min_pt = x_voo[idx_min, :].reshape((1, -1))
            n = x_voo.shape[0] - 1
            dists = r_sq(x_min_pt, np.delete(x_voo, [idx_min], axis=0), invlen=invlen)
            dist_min = np.min(dists)

            n_loop = 0
            while True:
                n_loop += 1
                if n_loop < n_loop_switch:
                    x_sel = x_minval + (x_maxval - x_minval) * rand((1, x_dim))
                else:
                    sigma = sigma_coef * np.sqrt(max(dist_min, 1e-12) / float(x_dim))
                    x_sel = x_min_pt + sigma * randn((1, x_dim))
                    x_sel = np.clip(x_sel, x_minval, x_maxval)
                dist_sel = r_sq(x_min_pt, x_sel, invlen=invlen)
                if dist_sel < dist_min or n_loop >= max_loop:
                    if n_loop >= max_loop:  # on failure, one global sample
                        x_sel = x_minval + (x_maxval - x_minval) * rand((1, x_dim))
                    break

        y_sel = evaluate_raw(x_sel, robot_chars=robot_chars, motion_type=motion_type,
                         pkl_path=pkl_path, file_path=file_path, animate=animate, repr_kind=repr_kind,
                         eval=eval, jnt_penalty=jnt_penalty, jnt_coeff=jnt_coeff, verbose=False)
        x_voo = np.concatenate((x_voo, x_sel), axis=0)
        y_voo = np.concatenate((y_voo, y_sel), axis=0)

    return np.asarray(x_voo,dtype=float).ravel(),np.asarray(y_voo, dtype=float).ravel()

# ── 2) model/seed setup ───────────────────────────────────────────────────────
n_runs = 10

# (optional) to use a completely different seed set per model, uncomment below and ignore the seeds above:
# base_seed_per_model = {'2dim_isoTrue':100000, '2dim_isoFalse':200000, '3dim_isoTrue':300000}

# ── 3) run ────────────────────────────────────────────────────────────────
runs_by_model = {'baseline': []}
tag='baseline'
runs_by_motion={}
runs_by_model[tag]=[]
for i in range(n_runs):
    # use shared seeds:
    rng=rngs[i]
    # (optional) to use independent seeds per model:
    # seed = base_seed_per_model[tag] + i

    x_seq,y_seq = run_raw_space_once(rng)
    runs_by_model[tag].append(y_seq)



In [ ]:
for key, value in enumerate(runs_by_model):
    print(f"keys: {key}, values: {value}")
    np.save(out(value+'VOO.npy'),runs_by_model[value])

In [ ]:
runs_by_model['baseline']

In [ ]:
# align lengths, then stack
T_min = min(min(len(y) for y in runs) for runs in runs_by_model.values())
print(f"T_min: {T_min}")
Y_by_model = {tag: np.stack([y[:T_min] for y in runs], axis=0)
              for tag, runs in runs_by_model.items()}

# statistics (mean / quartiles / min-max)
stats = {tag: {
            "mean": Y.mean(axis=0),
            "min":  Y.min(axis=0),
            "max":  Y.max(axis=0),
            "median": np.median(Y, axis=0),   # added (optional)
            "q25":  np.percentile(Y, 25, axis=0),
            "q75":  np.percentile(Y, 75, axis=0),
        } for tag, Y in Y_by_model.items()}

In [ ]:
stats['baseline']['mean']

In [ ]:
trying_baseline = np.load('baselineVOO.npy')
runs_by_model['baseline']=trying_baseline
trying_2f = np.load('2dim_isoFalseVOO.npy')
runs_by_model['2dim_isoFalse']=trying_2f
trying_2t = np.load('2dim_isoTrueVOO.npy')
runs_by_model['2dim_isoTrue']=trying_2t
trying_3t = np.load('3dim_isoTrueVOO.npy')
runs_by_model['3dim_isoTrue']=trying_3t


In [ ]:
np.min(stats['baseline']['best_mat'][:,-1])

In [ ]:
# ---- 1) summary: statistics over the full range with NaN padding for runs of different length ----
def summarize_best_full(runs):
    """runs: list of 1D arrays (y sequence of each run)."""
    T_max = max(len(y) for y in runs)
    B = np.full((len(runs), T_max), np.nan, float)
    for i, y in enumerate(runs):
        b = np.minimum.accumulate(np.asarray(y, float))
        B[i, :len(b)] = b
    stats = {
        "best_mat": B,                                    # (n_runs, T_max) with NaN pad
        "mean":   np.nanmean(B, axis=0),
        "median": np.nanmedian(B, axis=0),
        "min":    np.nanmin(B, axis=0),
        "max":    np.nanmax(B, axis=0),
        "q25":    np.nanpercentile(B, 25, axis=0),
        "q75":    np.nanpercentile(B, 75, axis=0),
        "count":  np.sum(~np.isnan(B), axis=0),           # number of valid runs per step
    }
    return stats

stats = {tag: summarize_best_full(runs) for tag, runs in runs_by_model.items()}

# ---- 2) x axis / axis limits / ticks ----
T_max_global = max(s["best_mat"].shape[1] for s in stats.values())
x = np.arange(1, T_max_global + 1)                       # 1-based index
ticks = np.arange(5, T_max_global + 1, 5)                # 5,10,15,...

# y axis tight to the 5~95% quantiles of all best-so-far values
all_best = np.concatenate([s["best_mat"].ravel() for s in stats.values()])
all_best = all_best[~np.isnan(all_best)]
yl, yh = np.percentile(all_best, [5, 95])
pad = 0.05 * (yh - yl)
yl, yh = yl - pad, yh + pad

# ---- 3) plot (categories in a separate column on the right) ----
fig, ax = plt.subplots(figsize=(10, 5))

# draw the mean (best-so-far) line
handles = {}
for tag in stats.keys():   # e.g. '2dim_isoTrue', '2dim_isoFalse', '3dim_isoTrue', 'baseline'
    h, = ax.plot(x, stats[tag]["mean"], label=tag, linewidth=2)
    handles[tag] = h

# vertical lines: every 5 steps, min-max as an I-bar (errorbar), IQR as a thick vline
offsets = np.linspace(-0.25, 0.25, num=len(stats))
cap_size_pts = 6
for (tag, off) in zip(stats.keys(), offsets):
    col = handles[tag].get_color()
    s = stats[tag]
    for t in ticks:
        i = t - 1
        lo, hi = s["min"][i], s["max"][i]
        ql, qh = s["q25"][i], s["q75"][i]
        cen = s["mean"][i]        # use s["median"][i] for the median
        # skip if there are no values (all NaN)
        if np.isnan(lo) or np.isnan(hi) or np.isnan(cen):
            continue
    viol_idx = np.where((s["mean"] < s["min"]) | (s["mean"] > s["max"]))[0]
    if viol_idx.size:
        print(f"[{tag}] violations at steps:", viol_idx + 1)  # 1-based
        print("counts:", s["count"][viol_idx])
        print("mean:", s["mean"][viol_idx])
        print("min :", s["min"][viol_idx])
        print("max :", s["max"][viol_idx])
        xi = t + off
        ax.errorbar(xi, cen, yerr=[[cen - lo], [hi - cen]],
                    fmt='none', ecolor=col, elinewidth=1,
                    capsize=cap_size_pts, capthick=1, alpha=0.5, zorder=2)
        if not (np.isnan(ql) or np.isnan(qh)):
            ax.vlines(xi, ql, qh, colors=col, linewidth=4, alpha=0.9, zorder=3)

ax.set_xlabel("Evaluation index")
ax.set_ylabel("Objective (best-so-far)")
ax.set_title("Mean of best-so-far with distribution bars every 5 evals")
ax.set_ylim(yl, yh)
ax.grid(True, alpha=0.3)

# ── place the legend in a column outside on the right
# shrink the plot area to 80% and pin the legend at the center right
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width * 0.78, box.height])
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True)

plt.show()

### Oracle value(data-based evaluation)

In [ ]:
len(robot_chars.classifying_joint_info.keys())

In [ ]:
model_spec = 'h64_32_z2_isoTrue_actvTanh_isoreg1e-07_nepoch1000_bs10'
dataset = robot_chars.create_data_to_classify(save=False)
sym_data = dataset.arrays[:,[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29], :].reshape(-1,120)
ae, config = load_saved_model(model_name=model_spec, device="cpu")
encode_try = ae.encode(th.tensor(sym_data,device=next(ae.parameters()).device,dtype=th.float32)).detach().numpy()

oracle_y={}
for i in range(len(dataset.labels)):
    print(f"encode_try[i]: {encode_try[i]}")
    y = evaluate(np.array([encode_try[i]]), ae, robot_chars, motion_type,
                     pkl_path, file_path, animate=animate, repr_kind=repr_kind,
                     eval=eval, jnt_penalty=jnt_penalty, jnt_coeff=jnt_coeff, verbose=False)
    print(f"robot name: {dataset.labels[i]}, encode_try: {encode_try[i]} value: {y}")
    oracle_y[dataset.labels[i]]=[encode_try[i,0],encode_try[i,1],y]
    



In [ ]:
real_value = []
for oracle_y_value, oracle_y_key in enumerate(oracle_y):
    # print(f"value: {oracle_y[oracle_y_key][-1][0][0]}")
    real_value.append(oracle_y[oracle_y_key][-1][0][0])
real_value
np.mean(real_value)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ---------------- Oracle: load values ----------------
# handle real_value whether it is a function or a list
ov = real_value() if callable(real_value) else real_value
oracle_vals = np.asarray(ov, float).ravel()

mu = float(np.mean(oracle_vals))
sigma = float(np.std(oracle_vals, ddof=1)) if np.std(oracle_vals, ddof=1) > 0 else 1e-12
q25, q75 = np.percentile(oracle_vals, [25, 75])
o_min, o_max = float(np.min(oracle_vals)), float(np.max(oracle_vals))

# ---------------- experiment statistics summary (full range, NaN padding) ----------------
def summarize_best_full(runs):
    T_max = max(len(y) for y in runs)
    B = np.full((len(runs), T_max), np.nan, float)
    for i, y in enumerate(runs):
        b = np.minimum.accumulate(np.asarray(y, float))
        B[i, :len(b)] = b
    return {
        "best_mat": B,
        "mean":   np.nanmean(B, axis=0),
        "min":    np.nanmin(B, axis=0),
        "max":    np.nanmax(B, axis=0),
        "q25":    np.nanpercentile(B, 25, axis=0),
        "q75":    np.nanpercentile(B, 75, axis=0),
    }

stats = {tag: summarize_best_full(runs) for tag, runs in runs_by_model.items()}
T_max_global = max(s["best_mat"].shape[1] for s in stats.values())
x = np.arange(1, T_max_global + 1)
ticks = np.arange(5, T_max_global + 1, 5)

# y-axis range (original units): consider both the oracle distribution and the experiment values
all_best = np.concatenate([s["best_mat"].ravel() for s in stats.values()])
all_best = all_best[~np.isnan(all_best)]
yl_raw = min(np.percentile(all_best, 5), o_min)
yh_raw = max(np.percentile(all_best, 95), o_max)
pad = 0.05 * (yh_raw - yl_raw)
yl_raw, yh_raw = yl_raw - pad, yh_raw + pad

# ---------------- Figure: left distribution + right main ----------------
fig = plt.figure(figsize=(11, 5))
gs = GridSpec(1, 2, width_ratios=[1.2, 8.8], wspace=0.05)

# (A) left: oracle distribution (horizontal histogram), y axis in original units
axL = fig.add_subplot(gs[0, 0])
axL.hist(oracle_vals, bins='auto', orientation='horizontal', alpha=0.30)
axL.axhspan(q25, q75, alpha=0.15)       # IQR band
axL.axhline(mu, linewidth=2)            # mean line
axL.set_ylim(yl_raw, yh_raw)
axL.set_xlabel("count")
axL.set_ylabel("Objective (raw)")
axL.set_title("Oracle distribution", fontsize=10)
axL.grid(True, axis='y', alpha=0.2)
axL.text(0.98, 0.02, f"μ={mu:.2f}\nσ={sigma:.2f}", ha='right', va='bottom',
         transform=axL.transAxes, fontsize=9)

# (B) right: experiment curves / vertical lines (original units), secondary y axis in -sigma units
ax = fig.add_subplot(gs[0, 1], sharey=axL)

# mean (best-so-far) line
handles = {}
for tag, s in stats.items():
    h, = ax.plot(x, s["mean"], label=tag, linewidth=2)
    handles[tag] = h

# I-bar (min~max) + thick IQR segment every 5 steps
offsets = np.linspace(-0.25, 0.25, num=len(stats))
for (tag, off) in zip(stats.keys(), offsets):
    col = handles[tag].get_color()
    s = stats[tag]
    for t in ticks:
        i = t - 1
        lo, hi = s["min"][i], s["max"][i]
        ql, qh = s["q25"][i], s["q75"][i]
        cen = s["mean"][i]
        if np.isnan(lo) or np.isnan(hi) or np.isnan(cen):
            continue
        xi = t + off
        ax.errorbar(xi, cen,
                    yerr=[[cen - lo], [hi - cen]],
                    fmt='none', ecolor=col, elinewidth=1,
                    capsize=6, capthick=1, alpha=0.5, zorder=2)
        if not (np.isnan(ql) or np.isnan(qh)):
            ax.vlines(xi, ql, qh, colors=col, linewidth=4, alpha=0.9, zorder=3)

ax.set_ylim(yl_raw, yh_raw)
ax.set_xlabel("Evaluation index (1-based)")
ax.set_ylabel("Objective (raw)")
ax.set_title("Mean of best-so-far (+ distribution bars)")

# secondary y axis: -sigma units (relative to the oracle)
def to_minus_sigma(y):      # raw -> Δ(−σ) = (μ - y)/σ
    return (mu - np.asarray(y)) / sigma
def from_minus_sigma(z):    # Δ(−σ) -> raw
    return mu - np.asarray(z) * sigma

ax_sigma = ax.secondary_yaxis('right', functions=(to_minus_sigma, from_minus_sigma))
ax_sigma.set_ylabel("Δ (−σ from oracle mean)")

# legend: column outside on the right
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width * 0.78, box.height])
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True)

plt.show()

In [ ]:
# x_voo[np.argmin(y_voo),0],x_voo[np.argmin(y_voo),1],y_voo[np.argmin(y_voo),0]

In [ ]:
import pandas as pd

cols = ['group','w_x','w_y','w_z','q_x','q_y','q_z']

def save_grouped_dict_as_csv(d: dict, path: str = 'kinematics_long.csv',
                             float_fmt: str = '%.6f'):
    """
    d: dict[str -> array_like of shape (N_i, 6)]
    path: output file path
    float_fmt: decimal format used when saving the CSV
    """
    records = []
    for group, arr in d.items():
        a = np.asarray(arr, dtype=float).reshape(-1, 6)  # guaranteed: (N_i, 6)
        for row in a:
            records.append((group, *row.tolist()))
    df = pd.DataFrame.from_records(records, columns=cols)
    df.to_csv(path, index=False, float_format=float_fmt)
    return df

# usage example
# data = {...}  # put your dict here as is.

# fix decimals not showing in Jupyter notebook display (display only):
pd.set_option('display.float_format', '{:.6f}'.format)

In [ ]:
# try_optimal = np.array([[7.179852691307394, -3.569850905850164]])
try_optimal = np.array([[7.17985253550413, -3.5698514289249914]])
decode_try = ae.decode(th.tensor(try_optimal,device=next(ae.parameters()).device,dtype=th.float32)).detach().numpy()
full_decode=robot_chars.build_full_interp_from_symmetric(interp_data_sym=decode_try,threshold=0.0,repr_kind=repr_kind)
data_to_visualize = robot_chars.convert_flat_interp_to_dict(full_decode,threshold=0.5,repr_kind=repr_kind)
clip = robot_chars.visualize_data_to_classify(data_to_visualize, save_video=True,vis=False, test = True, xml=True,repr_kind=repr_kind)
# env_to_view = MuJoCoParserClass(name='try2',xml_string=robot_chars.xml,verbose=False)
# env_to_view.open_interactive_viewer()
robot_chars.env.open_interactive_viewer()

# df = save_grouped_dict_as_csv(data_to_visualize['0'], 'kinematics_long.csv', float_fmt='%.6f')


In [ ]:
# robot_chars.explore_latent_dimension(data_np = data, model = ae,repr_kind=repr_kind)

### Visualize results

In [ ]:
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111)

# 2D Contour plot
contour = ax.contourf(x_1, x_2, z_1, levels=10, cmap='viridis')

# Data points (add here if any)
# ax.scatter(...)

# Labels and title
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title("VOO", fontsize=14)

# Colorbar
fig.colorbar(contour, ax=ax, label='Evaluation')

plt.tight_layout()
plt.show()

In [ ]:
easy_optimal = np.array([[-5.0757112738430035, 7.200598275418453]])
middle_optimal = np.array([[-7.120334, -8.1370581]])
hard_optimal = np.array([[8.13185319, -7.43121962]])


In [ ]:
robot_chars.explore_latent_dimension(data_np = data, model = ae,repr_kind=repr_kind)

In [ ]:
try_optimal = np.array([[-3.46,-0.09]])
decode_try = ae.decode(th.tensor(try_optimal,device=next(ae.parameters()).device,dtype=th.float32)).detach().numpy()
full_decode=robot_chars.build_full_interp_from_symmetric(interp_data_sym=decode_try,threshold=0.0,repr_kind=repr_kind)
data_to_visualize = robot_chars.convert_flat_interp_to_dict(full_decode,threshold=0.5,repr_kind=repr_kind)
file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/141/141_16.bvh' # easy
# file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/018/18_15.bvh' # middle
# file_path = '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/079/79_02.bvh' # hard

# file_path = '../../bvh/cmu-mocap/data/141/141_14.bvh' 
# file_path = '../../bvh/cmu-mocap/data/018/18_15.bvh' 
# file_path = '../../bvh/cmu-mocap/data/079/79_02.bvh' 

clip = robot_chars.visualize_data_to_classify(data_to_visualize, save_video=True,vis=True, test = False, xml=True,repr_kind=repr_kind)
# clip.ipython_display()
# print(f"robot_chars.xml: {robot_chars.xml['0']}")
env_try = MuJoCoParserClass(name='try', xml_string=robot_chars.xml['0'], verbose=False)
things_to_override_try = override_joi(env_try,repr_kind)
# print(env.body_names)
result = retarget_and_procrustes(
            env_try,
            pkl_path=pkl_path,
            file_path=file_path,
            motion_type = motion_type,
            ik_p_spine=False,
            ik_p_neck=True,
            leg_fixed=True,
            ik_R_hand=False,
            animate_chains = False,
            animate_ik=True,
            verbose=False,
            override= things_to_override_try,
            )

In [ ]:
import math
from pathlib import Path
import cv2
from PIL import Image
from IPython.display import display

_BACKENDS = {
    "auto": None,
    "ffmpeg": getattr(cv2, "CAP_FFMPEG", None),
    "gstreamer": getattr(cv2, "CAP_GSTREAMER", None),
    "msmf": getattr(cv2, "CAP_MSMF", None),
    "avfoundation": getattr(cv2, "CAP_AVFOUNDATION", None),
}

def display_frame_at(video_path, sec, resize_to=None, save_path=None,
                     method="nearest", backend="auto", tolerance_ms=None,
                     sequential_fallback=True):
    """
    Show the frame at a given time (sec) (returns PIL). Falls back to a sequential search if seeking fails.

    method: 'nearest' | 'floor' | 'ceil'
    backend: 'auto' | 'ffmpeg' | 'gstreamer' | 'msmf' | 'avfoundation'
    tolerance_ms: tolerance (ms) for deciding that seeking succeeded. If None, set to 1/fps*1.5.
    """
    be_id = _BACKENDS.get(backend, None)
    cap = cv2.VideoCapture(str(video_path)) if be_id is None else cv2.VideoCapture(str(video_path), be_id)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    try:
        backend_name = cap.getBackendName() if hasattr(cap, "getBackendName") else "unknown"
    except Exception:
        backend_name = "unknown"

    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration = (frame_count / fps) if (fps > 0 and frame_count > 0) else None

    if fps <= 0:
        # some backends return fps as 0. The MSEC/sequential fallback still handles it.
        fps = 30.0  # approximate
    if tolerance_ms is None:
        tolerance_ms = (1000.0 / fps) * 1.5

    # clamp time
    if duration is not None:
        sec = max(0.0, min(float(sec), max(0.0, duration - 1.0 / fps)))
    else:
        sec = max(0.0, float(sec))

    target_ms = sec * 1000.0
    frame = None

    def _ok_enough(current_ms):
        return abs((current_ms or 0.0) - target_ms) <= tolerance_ms

    # 1) try by frame index
    if frame_count > 0:
        idx = int({
            "floor": math.floor(sec * fps),
            "ceil":  math.ceil(sec * fps),
        }.get(method, round(sec * fps)))
        idx = max(0, min(idx, max(0, frame_count - 1)))
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, f = cap.read()
        if ok and f is not None:
            pos_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
            if _ok_enough(pos_ms):
                frame = f

    # 2) try by time (ms)
    if frame is None:
        cap.set(cv2.CAP_PROP_POS_MSEC, target_ms)
        ok, f = cap.read()
        if ok and f is not None:
            pos_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
            if _ok_enough(pos_ms):
                frame = f

    # 3) jump roughly by ratio, then search sequentially
    if frame is None and sequential_fallback:
        if duration:
            cap.set(cv2.CAP_PROP_POS_AVI_RATIO, max(0.0, min(0.9999, sec / duration)))
        # sequential read (exact). Choose nearest/floor/ceil by comparing the previous/current frame
        prev_f, prev_ms = None, None
        while True:
            ok, f = cap.read()
            if not ok or f is None:
                break
            cur_ms = cap.get(cv2.CAP_PROP_POS_MSEC)

            if method == "floor":
                if cur_ms >= target_ms:
                    frame = prev_f if prev_f is not None else f
                    break
            elif method == "ceil":
                if cur_ms >= target_ms:
                    frame = f
                    break
            else:  # nearest
                if cur_ms >= target_ms:
                    if prev_f is None:
                        frame = f
                    else:
                        frame = prev_f if abs(prev_ms - target_ms) <= abs(cur_ms - target_ms) else f
                    break

            prev_f, prev_ms = f, cur_ms

        # if nothing was picked even at the end, use the last valid frame
        if frame is None:
            frame = prev_f

    if frame is None:
        cap.release()
        raise RuntimeError(
            f"Seeking failed (backend={backend_name}). Force the FFmpeg backend (keep sequential_fallback=True) "
            f"Consider re-encoding with a different codec."
        )

    # BGR→RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # resize
    if resize_to is not None:
        w, h = resize_to
        frame_rgb = cv2.resize(frame_rgb, (w, h), interpolation=cv2.INTER_AREA)

    img = Image.fromarray(frame_rgb)

    # save options
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        img.save(str(save_path))

    # show
    display(img)
    cap.release()
    return img, {"backend": backend_name, "fps": fps, "frame_count": frame_count, "duration_sec": duration}

frame_sec=0.6
video_path = '141_16.mp4'
display_frame_at(video_path, frame_sec, resize_to=None, save_path=None, method='floor')

In [ ]:
decode = ae.decode(th.tensor(easy_optimal,device=next(ae.parameters()).device,dtype=th.float32)).detach().numpy()
full_decode=robot_chars.build_full_interp_from_symmetric(interp_data_sym=decode,threshold=0.5)
data_to_visualize = robot_chars.convert_flat_interp_to_dict(full_decode)

clip = robot_chars.visualize_data_to_classify(data_to_visualize, save_video=True,vis=True, test = True, xml=True)
clip.ipython_display()

In [ ]:
decode2 = ae.decode(th.tensor(middle_optimal,device=next(ae.parameters()).device,dtype=th.float32)).detach().numpy()
full_decode2=robot_chars.build_full_interp_from_symmetric(interp_data_sym=decode2,threshold=0.5,repr_kind=repr_kind)
data_to_visualize2 = robot_chars.convert_flat_interp_to_dict(full_decode2,repr_kind=repr_kind)

clip2 = robot_chars.visualize_data_to_classify(data_to_visualize2, save_video=True,vis=True, test=True, xml=True,repr_kind=repr_kind)
clip2.ipython_display()

In [ ]:
decode3 = ae.decode(th.tensor(hard_optimal,device=next(ae.parameters()).device,dtype=th.float32)).detach().numpy()
full_decode3=robot_chars.build_full_interp_from_symmetric(interp_data_sym=decode3,threshold=0.5)
data_to_visualize3 = robot_chars.convert_flat_interp_to_dict(full_decode3)

clip3 = robot_chars.visualize_data_to_classify(data_to_visualize3, save_video=True,vis=True,test=True, xml=True)
clip3.ipython_display()